In [2]:
# Cell 1
import re
import pandas as pd
from pathlib import Path

DATA_DIR = Path('.')
OUT_DIR  = Path('../data/samples')
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [66]:
# Cell 2: 특허 ID 정규화
def normalize_patent(s):
    if pd.isna(s): return None
    s = str(s).strip().upper()
    m = re.match(r'^([A-Z]*)(\d+)$', s)
    if not m: return None
    prefix, digits = m.group(1), m.group(2)
    return prefix + digits.lstrip('0')

In [67]:
# Cell 3: 원고/피고 분류 기준 + 간단 전처리 함수
PLAINTIFF_TYPES = {
    'Plaintiff', 'plaintiff',
    'Plaintiff Consolidated', 'Consolidated Plaintiff',
    'Consolidated Filer Plaintiff', 'Consol Plaintiff',
    'Plaintiff - Defaulted', 'Plaintiff, Counter Defendant',
}
DEFENDANT_TYPES = {
    'Defendant', 'defendant',
    'Consol Defendant', 'Defendant Consolidated', 'Consolidated Defendant',
}
COUNTER_AS_PLAINTIFF = {
    'Counter Defendant', 'Counter-defendant', 'Counter-Defendant',
    'counter-defendant', 'Counter Defendant Consolidated',
    'Consol Counter Defendant', 'Consol CounterDft',
    'Counterclaim Defendant', 'Consolidated Counter Defendant',
}
COUNTER_AS_DEFENDANT = {
    'Counter Claimant', 'Counter-claimant', 'Counter-Claimant',
    'counter-claimant', 'Counter Claimant Consolidated',
    'Consol Counter Claimant', 'Consol CounterCla',
    'Counter Plaintiff', 'Counter-Plaintiff', 'Counterclaim Plaintiff',
    'Consol Counter Plaintiff',
}

def classify_side(pt):
    if pt in PLAINTIFF_TYPES:      return 'P'
    if pt in DEFENDANT_TYPES:      return 'D'
    if pt in COUNTER_AS_PLAINTIFF: return 'CP'
    if pt in COUNTER_AS_DEFENDANT: return 'CD'
    return None

def simple_norm(s):
    if pd.isna(s): return ''
    s = str(s).strip().lower()
    s = re.sub(r'[,\.\-\(\)\/\\&\'"]+', ' ', s)
    s = re.sub(r'\s{2,}', ' ', s).strip()
    return s

In [68]:
# Cell 4: 데이터 로드 및 케이스별 집계
print('데이터 로드 중...')
inf_cases = pd.read_csv(DATA_DIR / 'inf_cases_with_patents.csv')
names     = pd.read_csv(DATA_DIR / 'names.csv')
patents   = pd.read_csv(DATA_DIR / 'patents.csv', low_memory=False)

# 특허 타입 필터 + 정규화
lit_patents = patents[
    patents['case_row_id'].isin(inf_cases['case_row_id']) &
    (patents['patent_doc_type'].str.strip() == 'Patent')
].copy()
lit_patents['patent_norm'] = lit_patents['patent'].astype(str).apply(normalize_patent)
lit_patents = lit_patents[lit_patents['patent_norm'].notna()]

# 케이스별 특허 집계
patents_agg = (
    lit_patents
    .groupby('case_row_id')['patent_norm']
    .apply(lambda x: ' | '.join(sorted(x.dropna().unique())))
    .rename('patents')
    .reset_index()
)

# 원고/피고 분류 — simple_norm 기준으로 dedup
lit_names = names[names['case_row_id'].isin(inf_cases['case_row_id'])].copy()
lit_names['side'] = lit_names['party_type'].apply(classify_side)
lit_names = lit_names[lit_names['side'].notna() & lit_names['name'].notna()]

def aggregate_parties(group):
    classified = {}   # key: simple_norm(name) → value: (side, raw_name)
    for _, row in group.iterrows():
        name = row['name'].strip()
        if not name: continue
        norm_key = simple_norm(name)
        if not norm_key or norm_key in classified:
            continue
        if row['side'] in ('P', 'CP'):
            classified[norm_key] = ('P', name)
        elif row['side'] in ('D', 'CD'):
            classified[norm_key] = ('D', name)
    plaintiffs = sorted(v[1] for v in classified.values() if v[0] == 'P')
    defendants = sorted(v[1] for v in classified.values() if v[0] == 'D')
    return pd.Series({
        'plaintiffs': ' | '.join(plaintiffs) if plaintiffs else None,
        'defendants': ' | '.join(defendants) if defendants else None,
    })

print('원고/피고 집계 중...')
parties = (
    lit_names
    .groupby('case_row_id')
    .apply(aggregate_parties, include_groups=False)
    .reset_index()
)

# 최종 케이스 테이블
case_table = (
    inf_cases[['case_row_id', 'case_number', 'date_filed', 'court_name', 'district_id']]
    .merge(parties,     on='case_row_id', how='inner')
    .merge(patents_agg, on='case_row_id', how='inner')
    .dropna(subset=['plaintiffs', 'defendants', 'patents'])
    .reset_index(drop=True)
)

print(f'케이스 수: {len(case_table):,}건')
print(f'고유 특허: {lit_patents["patent_norm"].nunique():,}개')
case_table.head(3)

데이터 로드 중...
원고/피고 집계 중...
케이스 수: 45,950건
고유 특허: 37,806개


,case_row_id,case_number,date_filed,court_name,district_id,plaintiffs,defendants,patents
0,727,0:03-cv-00968,2003-01-30,U.S. District of Minnesota (DMN),mnd,"Magnum Research, Inc",Advanced Barrel Systems | Michael Degerness | ...,5600912 | 6457274
1,728,0:03-cv-00993,2003-02-03,U.S. District of Minnesota (DMN),mnd,"Crestliner, Inc.",Alumacraft Boat Co,5108321
2,729,0:03-cv-01016,2003-02-04,U.S. District of Minnesota (DMN),mnd,Taylor Corporation,"AS Hospitality, Inc.",6153278


In [69]:
# Cell 5: 원고=피고 겹침 확인 + 총 케이스-특허 행 수
def has_overlap(row):
    if pd.isna(row['plaintiffs']) or pd.isna(row['defendants']):
        return False
    pla_norms = {simple_norm(p) for p in row['plaintiffs'].split(' | ') if simple_norm(p)}
    def_norms = {simple_norm(d) for d in row['defendants'].split(' | ') if simple_norm(d)}
    return bool(pla_norms & def_norms)

overlap_case_ids = set(case_table[case_table.apply(has_overlap, axis=1)]['case_row_id'])
print(f'원고=피고 겹치는 케이스: {len(overlap_case_ids):,}건')

n = (
    lit_patents[lit_patents['case_row_id'].isin(case_table['case_row_id'])]
    [['case_row_id', 'patent_norm']]
    .drop_duplicates()
    .shape[0]
)
print(f'총 케이스-특허 행 수: {n:,}행')

원고=피고 겹치는 케이스: 0건
총 케이스-특허 행 수: 101,802행


In [70]:
# Cell 6: g_assignee_disambiguated 로드 → 소송 특허 필터
litigated_set = set(lit_patents['patent_norm'].dropna().unique())
print(f'소송 특허 고유 수: {len(litigated_set):,}개')

print('g_assignee_disambiguated.tsv 로딩 중...')
chunks = []
for chunk in pd.read_csv(
    DATA_DIR / 'g_assignee_disambiguated.tsv', sep='\t',
    usecols=['patent_id', 'disambig_assignee_organization'],
    chunksize=500_000
):
    chunk['patent_id'] = chunk['patent_id'].astype(str)
    chunk['patent_norm'] = chunk['patent_id'].apply(normalize_patent)
    matched = chunk[chunk['patent_norm'].isin(litigated_set)]
    if len(matched):
        chunks.append(matched)

g_asgn = (
    pd.concat(chunks, ignore_index=True)
    .pipe(lambda df: df[df['disambig_assignee_organization'].notna()])
    [['patent_norm', 'disambig_assignee_organization']]
    .rename(columns={'disambig_assignee_organization': 'assignee_org'})
    .drop_duplicates()
    .reset_index(drop=True)
)
print(f'로드 완료: {len(g_asgn):,}건')
print(f'고유 특허: {g_asgn["patent_norm"].nunique():,}개')

patent_to_assignees = g_asgn.groupby('patent_norm')['assignee_org'].apply(list).to_dict()

소송 특허 고유 수: 37,806개
g_assignee_disambiguated.tsv 로딩 중...
로드 완료: 30,725건
고유 특허: 30,252개


In [71]:
# Cell 7: Step 2 — Raw 매칭 (대소문자 무시, 구두점 유지)
def match_raw_all(assignees, party_str):
    if not isinstance(assignees, list) or not assignees or pd.isna(party_str):
        return '', ''
    party_map = {p.strip().lower(): p.strip() for p in party_str.split(' | ')}
    for org in assignees:
        if pd.isna(org): continue
        if org.strip().lower() in party_map:
            return org.strip(), party_map[org.strip().lower()]
    return '', ''

# 케이스-특허 단위 DataFrame
case_pat = (
    lit_patents[['case_row_id', 'patent_norm']]
    .drop_duplicates()
    .merge(case_table[['case_row_id', 'date_filed', 'plaintiffs', 'defendants']],
           on='case_row_id', how='left')
)
case_pat['assignees'] = case_pat['patent_norm'].map(patent_to_assignees)

results = case_pat.apply(
    lambda r: pd.Series({
        'raw_matched_assignee':   match_raw_all(r['assignees'], r['plaintiffs'])[0],
        'raw_pla_match':          match_raw_all(r['assignees'], r['plaintiffs'])[1],
        'raw_matched_assignee_d': match_raw_all(r['assignees'], r['defendants'])[0],
        'raw_def_match':          match_raw_all(r['assignees'], r['defendants'])[1],
    }), axis=1
)
case_pat = pd.concat([case_pat, results], axis=1)

case_pat['match_src'] = 'unmatched'
case_pat.loc[~case_pat['assignees'].apply(lambda x: isinstance(x, list)), 'match_src'] = 'no_assignee'
case_pat.loc[case_pat['raw_pla_match'] != '', 'match_src'] = 'raw_plaintiff'
case_pat.loc[(case_pat['raw_pla_match'] == '') & (case_pat['raw_def_match'] != ''), 'match_src'] = 'raw_defendant'

total = len(case_pat)
print('=== Step 2: Raw 매칭 결과 ===')
for src, label in [
    ('raw_plaintiff', '원고 raw 매칭'),
    ('raw_defendant', '피고 raw 매칭'),
    ('unmatched',     'assignee 있으나 매칭 실패'),
    ('no_assignee',   'g_assignee 데이터 없음'),
]:
    n = (case_pat['match_src'] == src).sum()
    print(f'  {label:30s}: {n:>7,}행 ({n/total:.1%})')

=== Step 2: Raw 매칭 결과 ===
  원고 raw 매칭                     :  21,279행 (20.9%)
  피고 raw 매칭                     :     648행 (0.6%)
  assignee 있으나 매칭 실패            :  59,706행 (58.6%)
  g_assignee 데이터 없음             :  20,285행 (19.9%)


In [72]:
# Cell 8: Step 3 — 간단 전처리 후 재매칭 (unmatched만)
def match_simple_all(assignees, party_str):
    if not isinstance(assignees, list) or not assignees or pd.isna(party_str):
        return '', ''
    party_map = {simple_norm(p): p.strip() for p in party_str.split(' | ')}
    party_map.pop('', None)
    for org in assignees:
        if pd.isna(org): continue
        key = simple_norm(org)
        if key and key in party_map:
            return org.strip(), party_map[key]
    return '', ''

unmatched_mask = case_pat['match_src'] == 'unmatched'
simple_results = case_pat[unmatched_mask].apply(
    lambda r: pd.Series({
        'simple_matched_assignee':   match_simple_all(r['assignees'], r['plaintiffs'])[0],
        'simple_pla_match':          match_simple_all(r['assignees'], r['plaintiffs'])[1],
        'simple_matched_assignee_d': match_simple_all(r['assignees'], r['defendants'])[0],
        'simple_def_match':          match_simple_all(r['assignees'], r['defendants'])[1],
    }), axis=1
)
for col in simple_results.columns:
    case_pat[col] = simple_results[col].reindex(case_pat.index).fillna('')

case_pat.loc[(case_pat['match_src'] == 'unmatched') & (case_pat['simple_pla_match'] != ''),
             'match_src'] = 'simple_plaintiff'
case_pat.loc[(case_pat['match_src'] == 'unmatched') &
             (case_pat['simple_pla_match'] == '') & (case_pat['simple_def_match'] != ''),
             'match_src'] = 'simple_defendant'

total = len(case_pat)
print('=== Step 3: 전처리 후 누계 ===')
for src, label in [
    ('raw_plaintiff',    'Raw      → 원고'),
    ('raw_defendant',    'Raw      → 피고'),
    ('simple_plaintiff', '전처리   → 원고 추가'),
    ('simple_defendant', '전처리   → 피고 추가'),
    ('unmatched',        'assignee 있으나 미매칭'),
    ('no_assignee',      'g_assignee 없음'),
]:
    n = (case_pat['match_src'] == src).sum()
    print(f'  {label:28s}: {n:>7,}행 ({n/total:.1%})')

matched = case_pat['match_src'].isin(['raw_plaintiff','raw_defendant','simple_plaintiff','simple_defendant']).sum()
print(f'\n  누계 매칭: {matched:,}행 ({matched/total:.1%})')

=== Step 3: 전처리 후 누계 ===
  Raw      → 원고               :  21,279행 (20.9%)
  Raw      → 피고               :     648행 (0.6%)
  전처리   → 원고 추가               :   9,955행 (9.8%)
  전처리   → 피고 추가               :     256행 (0.3%)
  assignee 있으나 미매칭            :  49,495행 (48.6%)
  g_assignee 없음               :  20,285행 (19.9%)

  누계 매칭: 32,138행 (31.5%)


In [73]:
# Cell 9: Step 4 — Assignment Chain 이력 로드
OWNERSHIP_TYPES = {'assignment', 'namechg', 'merger', 'employee'}

unmatched_patents = set(case_pat[case_pat['match_src'] == 'unmatched']['patent_norm'].unique())
print(f'chain 대상 특허: {len(unmatched_patents):,}개')

print('assignment_conveyance.csv 로드...')
ac = pd.read_csv(DATA_DIR / 'assignment_conveyance.csv', usecols=['rf_id', 'convey_ty'])
valid_rfids = set(ac.loc[ac['convey_ty'].isin(OWNERSHIP_TYPES), 'rf_id'])
del ac

print('assignment.csv 로드...')
asgn = pd.read_csv(DATA_DIR / 'assignment.csv',
                   usecols=['rf_id', 'record_dt', 'purge_in'], low_memory=False)
asgn = asgn[asgn['rf_id'].isin(valid_rfids) & (asgn['purge_in'] == 'N')]
asgn['record_dt'] = pd.to_datetime(asgn['record_dt'], errors='coerce')
asgn = asgn.dropna(subset=['record_dt']).drop_duplicates('rf_id')[['rf_id', 'record_dt']]
valid_rfids2 = set(asgn['rf_id'])

print('assignee.csv 로드...')
ee = pd.read_csv(DATA_DIR / 'assignee.csv', usecols=['rf_id', 'ee_name'], low_memory=False)
ee = ee[ee['rf_id'].isin(valid_rfids2) & ee['ee_name'].notna()].drop_duplicates('rf_id')

print('documentid.csv 로드...')
doc = pd.read_csv(DATA_DIR / 'documentid.csv',
                  usecols=['rf_id', 'grant_doc_num'], low_memory=False)
doc = doc[doc['rf_id'].isin(valid_rfids2)].dropna(subset=['grant_doc_num']).copy()
doc['patent_norm'] = doc['grant_doc_num'].astype(str).str.strip().apply(normalize_patent)
doc = doc[doc['patent_norm'].isin(unmatched_patents)][['rf_id', 'patent_norm']]

chain_df = (
    doc.merge(asgn, on='rf_id').merge(ee, on='rf_id')
    [['patent_norm', 'record_dt', 'ee_name']]
    .drop_duplicates()
    .sort_values(['patent_norm', 'record_dt'])
    .reset_index(drop=True)
)
INF_DATE = pd.Timestamp('2099-12-31')
chain_df['end_dt'] = chain_df.groupby('patent_norm')['record_dt'].shift(-1).fillna(INF_DATE)
chain_df.rename(columns={'record_dt': 'start_dt'}, inplace=True)
print(f'chain_df: {len(chain_df):,}행 / 고유 특허: {chain_df["patent_norm"].nunique():,}개')

chain 대상 특허: 16,591개
assignment_conveyance.csv 로드...
assignment.csv 로드...
assignee.csv 로드...
documentid.csv 로드...
chain_df: 42,343행 / 고유 특허: 15,281개


In [74]:
# Cell 10: Step 4 — Chain 구간 매칭 (date_filed ~ date_last_filed)
inf_dates = pd.read_csv(DATA_DIR / 'inf_cases_with_patents.csv',
                        usecols=['case_row_id', 'date_last_filed'])
case_pat = case_pat.merge(inf_dates, on='case_row_id', how='left')
case_pat['date_filed_dt']      = pd.to_datetime(case_pat['date_filed'],      errors='coerce')
case_pat['date_last_filed_dt'] = pd.to_datetime(case_pat['date_last_filed'], errors='coerce')
case_pat['date_last_filed_dt'] = case_pat['date_last_filed_dt'].fillna(case_pat['date_filed_dt'])

cp_un = case_pat[case_pat['match_src'] == 'unmatched'][
    ['case_row_id', 'patent_norm', 'date_filed_dt', 'date_last_filed_dt']
].copy()

cp_chain = cp_un.merge(chain_df, on='patent_norm', how='left')
overlap = (
    cp_chain['start_dt'].notna() &
    (cp_chain['start_dt'] < cp_chain['date_last_filed_dt']) &
    (cp_chain['end_dt']   > cp_chain['date_filed_dt'])
)
chain_owners = (
    cp_chain[overlap]
    .groupby(['case_row_id', 'patent_norm'])['ee_name']
    .apply(lambda x: list(x.dropna().unique()))
    .rename('chain_assignees')
    .reset_index()
)
case_pat = case_pat.merge(chain_owners, on=['case_row_id', 'patent_norm'], how='left')
if 'chain_assignees' not in case_pat.columns:
    case_pat['chain_assignees'] = None

# raw 매칭
unmatched_mask = case_pat['match_src'] == 'unmatched'
chain_raw = case_pat[unmatched_mask].apply(
    lambda r: pd.Series({
        'chain_raw_pla_match': match_raw_all(r['chain_assignees'], r['plaintiffs'])[1],
        'chain_raw_def_match': match_raw_all(r['chain_assignees'], r['defendants'])[1],
        'chain_raw_owner_pla': match_raw_all(r['chain_assignees'], r['plaintiffs'])[0],
        'chain_raw_owner_def': match_raw_all(r['chain_assignees'], r['defendants'])[0],
    }), axis=1
)
for col in chain_raw.columns:
    case_pat[col] = chain_raw[col].reindex(case_pat.index).fillna('')

# simple 전처리 매칭
need_simple = unmatched_mask & (case_pat['chain_raw_pla_match'] == '') & \
              (case_pat['chain_raw_def_match'] == '')
chain_simple = case_pat[need_simple].apply(
    lambda r: pd.Series({
        'chain_simple_pla_match': match_simple_all(r['chain_assignees'], r['plaintiffs'])[1],
        'chain_simple_def_match': match_simple_all(r['chain_assignees'], r['defendants'])[1],
        'chain_simple_owner_pla': match_simple_all(r['chain_assignees'], r['plaintiffs'])[0],
        'chain_simple_owner_def': match_simple_all(r['chain_assignees'], r['defendants'])[0],
    }), axis=1
)
for col in chain_simple.columns:
    case_pat[col] = chain_simple[col].reindex(case_pat.index).fillna('')

case_pat.loc[unmatched_mask & (case_pat['chain_raw_pla_match'] != ''),
             'match_src'] = 'chain_raw_plaintiff'
case_pat.loc[unmatched_mask & (case_pat['chain_raw_pla_match'] == '') &
             (case_pat['chain_raw_def_match'] != ''),
             'match_src'] = 'chain_raw_defendant'
case_pat.loc[unmatched_mask & (case_pat['chain_simple_pla_match'] != ''),
             'match_src'] = 'chain_simple_plaintiff'
case_pat.loc[unmatched_mask & (case_pat['chain_simple_pla_match'] == '') &
             (case_pat['chain_simple_def_match'] != ''),
             'match_src'] = 'chain_simple_defendant'

total = len(case_pat)
print('=== Step 4: Chain 구간 매칭 누계 ===')
for src, label in [
    ('raw_plaintiff',          'g_assignee Raw    → 원고'),
    ('raw_defendant',          'g_assignee Raw    → 피고'),
    ('simple_plaintiff',       'g_assignee 전처리 → 원고'),
    ('simple_defendant',       'g_assignee 전처리 → 피고'),
    ('chain_raw_plaintiff',    'Chain Raw         → 원고'),
    ('chain_raw_defendant',    'Chain Raw         → 피고'),
    ('chain_simple_plaintiff', 'Chain 전처리      → 원고'),
    ('chain_simple_defendant', 'Chain 전처리      → 피고'),
    ('unmatched',              '최종 미매칭'),
    ('no_assignee',            'g_assignee 없음'),
]:
    n = (case_pat['match_src'] == src).sum()
    print(f'  {label:30s}: {n:>7,}행 ({n/total:.1%})')

matched = case_pat['match_src'].str.contains('plaintiff|defendant', na=False).sum()
print(f'\n  전체 매칭: {matched:,}행 ({matched/total:.1%})')

=== Step 4: Chain 구간 매칭 누계 ===
  g_assignee Raw    → 원고        :  21,279행 (20.9%)
  g_assignee Raw    → 피고        :     648행 (0.6%)
  g_assignee 전처리 → 원고           :   9,955행 (9.8%)
  g_assignee 전처리 → 피고           :     256행 (0.3%)
  Chain Raw         → 원고        :  17,519행 (17.2%)
  Chain Raw         → 피고        :     384행 (0.4%)
  Chain 전처리      → 원고           :   9,332행 (9.2%)
  Chain 전처리      → 피고           :     157행 (0.2%)
  최종 미매칭                        :  22,103행 (21.7%)
  g_assignee 없음                 :  20,285행 (19.9%)

  전체 매칭: 59,530행 (58.4%)


In [75]:
# Cell 12: no_assignee 특허도 Chain 매칭 시도
no_asgn_patents = set(case_pat[case_pat['match_src'] == 'no_assignee']['patent_norm'].unique())
print(f'no_assignee 대상 특허: {len(no_asgn_patents):,}개')

print('documentid.csv에서 no_assignee 특허 검색...')
doc2 = pd.read_csv(DATA_DIR / 'documentid.csv',
                   usecols=['rf_id', 'grant_doc_num'], low_memory=False)
doc2 = doc2[doc2['rf_id'].isin(valid_rfids2)].dropna(subset=['grant_doc_num']).copy()
doc2['patent_norm'] = doc2['grant_doc_num'].astype(str).str.strip().apply(normalize_patent)
doc2 = doc2[doc2['patent_norm'].isin(no_asgn_patents)][['rf_id', 'patent_norm']]
print(f'  매핑된 rf_id: {len(doc2):,}개')

chain_df2 = (
    doc2.merge(asgn, on='rf_id').merge(ee, on='rf_id')
    [['patent_norm', 'record_dt', 'ee_name']]
    .drop_duplicates()
    .sort_values(['patent_norm', 'record_dt'])
    .reset_index(drop=True)
)
chain_df2['end_dt'] = chain_df2.groupby('patent_norm')['record_dt'].shift(-1).fillna(INF_DATE)
chain_df2.rename(columns={'record_dt': 'start_dt'}, inplace=True)
print(f'  chain_df2: {len(chain_df2):,}행 / 고유 특허: {chain_df2["patent_norm"].nunique():,}개')

no_asgn_mask = case_pat['match_src'] == 'no_assignee'
cp_na = case_pat[no_asgn_mask][
    ['case_row_id', 'patent_norm', 'date_filed_dt', 'date_last_filed_dt']
].copy()

cp_chain2 = cp_na.merge(chain_df2, on='patent_norm', how='left')
overlap2 = (
    cp_chain2['start_dt'].notna() &
    (cp_chain2['start_dt'] < cp_chain2['date_last_filed_dt']) &
    (cp_chain2['end_dt']   > cp_chain2['date_filed_dt'])
)
chain_owners2 = (
    cp_chain2[overlap2]
    .groupby(['case_row_id', 'patent_norm'])['ee_name']
    .apply(lambda x: list(x.dropna().unique()))
    .rename('chain_assignees2')
    .reset_index()
)
case_pat = case_pat.merge(chain_owners2, on=['case_row_id', 'patent_norm'], how='left')
if 'chain_assignees2' not in case_pat.columns:
    case_pat['chain_assignees2'] = None

na_chain_raw = case_pat[no_asgn_mask].apply(
    lambda r: pd.Series({
        'na_chain_raw_pla_match': match_raw_all(r['chain_assignees2'], r['plaintiffs'])[1],
        'na_chain_raw_def_match': match_raw_all(r['chain_assignees2'], r['defendants'])[1],
        'na_chain_raw_owner_pla': match_raw_all(r['chain_assignees2'], r['plaintiffs'])[0],
        'na_chain_raw_owner_def': match_raw_all(r['chain_assignees2'], r['defendants'])[0],
    }), axis=1
)
for col in na_chain_raw.columns:
    case_pat[col] = na_chain_raw[col].reindex(case_pat.index).fillna('')

need_simple2 = no_asgn_mask & (case_pat['na_chain_raw_pla_match'] == '') & \
               (case_pat['na_chain_raw_def_match'] == '')
na_chain_simple = case_pat[need_simple2].apply(
    lambda r: pd.Series({
        'na_chain_simple_pla_match': match_simple_all(r['chain_assignees2'], r['plaintiffs'])[1],
        'na_chain_simple_def_match': match_simple_all(r['chain_assignees2'], r['defendants'])[1],
        'na_chain_simple_owner_pla': match_simple_all(r['chain_assignees2'], r['plaintiffs'])[0],
        'na_chain_simple_owner_def': match_simple_all(r['chain_assignees2'], r['defendants'])[0],
    }), axis=1
)
for col in na_chain_simple.columns:
    case_pat[col] = na_chain_simple[col].reindex(case_pat.index).fillna('')

case_pat.loc[no_asgn_mask & (case_pat['na_chain_raw_pla_match'] != ''),
             'match_src'] = 'na_chain_raw_plaintiff'
case_pat.loc[no_asgn_mask & (case_pat['na_chain_raw_pla_match'] == '') &
             (case_pat['na_chain_raw_def_match'] != ''),
             'match_src'] = 'na_chain_raw_defendant'
case_pat.loc[no_asgn_mask & (case_pat['na_chain_simple_pla_match'] != ''),
             'match_src'] = 'na_chain_simple_plaintiff'
case_pat.loc[no_asgn_mask & (case_pat['na_chain_simple_pla_match'] == '') &
             (case_pat['na_chain_simple_def_match'] != ''),
             'match_src'] = 'na_chain_simple_defendant'

total = len(case_pat)
na_matched = case_pat['match_src'].str.startswith('na_chain', na=False).sum()
print(f'\nno_assignee 중 chain으로 추가 매칭: {na_matched:,}행 ({na_matched/total:.1%})')
matched_all = case_pat['match_src'].str.contains('plaintiff|defendant', na=False).sum()
print(f'전체 누계 매칭: {matched_all:,}행 ({matched_all/total:.1%})')

no_assignee 대상 특허: 7,554개
documentid.csv에서 no_assignee 특허 검색...
  매핑된 rf_id: 10,676개
  chain_df2: 10,297행 / 고유 특허: 5,035개

no_assignee 중 chain으로 추가 매칭: 9,887행 (9.7%)
전체 누계 매칭: 69,417행 (68.1%)


In [76]:
# Cell 13: 최종 결과 CSV 저장 + 단계별 통계
case_pat['final_owner']      = ''
case_pat['matched_assignee'] = ''

src_map = {
    'raw_plaintiff':              ('raw_pla_match',              'raw_matched_assignee'),
    'raw_defendant':              ('raw_def_match',               'raw_matched_assignee_d'),
    'simple_plaintiff':           ('simple_pla_match',            'simple_matched_assignee'),
    'simple_defendant':           ('simple_def_match',            'simple_matched_assignee_d'),
    'chain_raw_plaintiff':        ('chain_raw_pla_match',         'chain_raw_owner_pla'),
    'chain_raw_defendant':        ('chain_raw_def_match',         'chain_raw_owner_def'),
    'chain_simple_plaintiff':     ('chain_simple_pla_match',      'chain_simple_owner_pla'),
    'chain_simple_defendant':     ('chain_simple_def_match',      'chain_simple_owner_def'),
    'na_chain_raw_plaintiff':     ('na_chain_raw_pla_match',      'na_chain_raw_owner_pla'),
    'na_chain_raw_defendant':     ('na_chain_raw_def_match',      'na_chain_raw_owner_def'),
    'na_chain_simple_plaintiff':  ('na_chain_simple_pla_match',   'na_chain_simple_owner_pla'),
    'na_chain_simple_defendant':  ('na_chain_simple_def_match',   'na_chain_simple_owner_def'),
}
for src, (party_col, owner_col) in src_map.items():
    mask = case_pat['match_src'] == src
    if mask.any() and party_col in case_pat.columns:
        case_pat.loc[mask, 'final_owner']      = case_pat.loc[mask, party_col]
        case_pat.loc[mask, 'matched_assignee'] = case_pat.loc[mask, owner_col]

case_pat['owner_side'] = ''
case_pat.loc[case_pat['match_src'].str.contains('plaintiff', na=False), 'owner_side'] = 'P'
case_pat.loc[case_pat['match_src'].str.contains('defendant', na=False), 'owner_side'] = 'D'

out_cols = ['case_row_id', 'date_filed', 'date_last_filed', 'patent_norm',
            'matched_assignee', 'owner_side', 'final_owner', 'match_src',
            'plaintiffs', 'defendants']
out_path = OUT_DIR / 'patent_ownership_matched_2.csv'
case_pat[out_cols].to_csv(out_path, index=False)
print(f'저장 완료: {out_path}')
print()

total = len(case_pat)
step_groups = [
    ('Step 2 Raw',      ['raw_plaintiff',           'raw_defendant']),
    ('Step 3 전처리',   ['simple_plaintiff',         'simple_defendant']),
    ('Step 4 Chain',    ['chain_raw_plaintiff',      'chain_raw_defendant',
                         'chain_simple_plaintiff',   'chain_simple_defendant']),
    ('Step 5 NA Chain', ['na_chain_raw_plaintiff',   'na_chain_raw_defendant',
                         'na_chain_simple_plaintiff','na_chain_simple_defendant']),
]
print(f'{"단계":<18} {"이번 단계":>10} {"누적":>10} {"누적 비율":>10}')
print('-' * 52)
cumulative = 0
for label, srcs in step_groups:
    n = case_pat['match_src'].isin(srcs).sum()
    cumulative += n
    print(f'{label:<18} {n:>10,} {cumulative:>10,} {cumulative/total:>9.1%}')

n_un = (case_pat['match_src'] == 'unmatched').sum()
n_na = (case_pat['match_src'] == 'no_assignee').sum()
print('-' * 52)
print(f'{"최종 미매칭":<18} {n_un:>10,} {"": >10} {n_un/total:>9.1%}')
print(f'{"g_assignee 없음":<18} {n_na:>10,} {"": >10} {n_na/total:>9.1%}')
print(f'{"합계":<18} {total:>10,}')

저장 완료: ../data/samples/patent_ownership_matched_2.csv

단계                      이번 단계         누적      누적 비율
----------------------------------------------------
Step 2 Raw             21,927     21,927     21.5%
Step 3 전처리             10,211     32,138     31.5%
Step 4 Chain           27,392     59,530     58.4%
Step 5 NA Chain         9,887     69,417     68.1%
----------------------------------------------------
최종 미매칭                 22,103                21.7%
g_assignee 없음          10,398                10.2%
합계                    101,918


In [77]:
case_pat[case_pat['match_src'].isin(['unmatched', 'no_assignee'])]['patent_norm'].drop_duplicates().head(20).tolist()

['6457274',
 '6411942',
 '6285752',
 '6393587',
 '5196040',
 '6312197',
 '6507923',
 '6032723',
 '5277849',
 '6241024',
 '6395702',
 '6361499',
 '6293475',
 '6158668',
 '4908016',
 '5006344',
 '4654217',
 '5725564',
 '6461284',
 '5203048']

In [80]:
unmatched_df = case_pat[case_pat['match_src'] == 'unmatched'].copy()

unmatched_df['g_assignee_owners'] = unmatched_df['assignees'].apply(
    lambda x: ' | '.join(x) if isinstance(x, list) else ''
)
unmatched_df['chain_owners'] = unmatched_df['chain_assignees'].apply(
    lambda x: ' | '.join(x) if isinstance(x, list) else ''
)

out = unmatched_df[['case_row_id', 'patent_norm', 'g_assignee_owners', 'chain_owners',
                     'plaintiffs', 'defendants']]

out.to_csv(OUT_DIR / 'unmatched_review.csv', index=False)
print(f'저장 완료: {len(out):,}행')
out.head(5)

저장 완료: 22,103행


,case_row_id,patent_norm,g_assignee_owners,chain_owners,plaintiffs,defendants
12,742,6411942,Fujitsu Limited,FUJITSU LIMITED,"Firepond, Inc. | a Delaware corporation","Banter, Inc."
16,747,6393587,I-Tech Corp.,,I-Tech Corporation | a Minnesota corporation,Ancot Corporation | a California corporation
19,750,5196040,Dundee Manufacturing Co. Inc. | Grease Guard LLC,"GREASE GUARD, INC.","Ecolab, Inc. | Facilitec Corp.","Omni Containment Systems, LLC"
22,753,6312197,"Anchor Wall Systems, Inc.",,Anchor Wall Systems | a Minnesota corporation,"Concrete Products of New London, Inc."
24,756,6507923,I-Tech Corp.,I-TECH CORP. | FINISAR CORPORATION,I-Tech Corporation | a Minnesota corporation,Computer Bus Tools | Data Transit Corporation ...


In [81]:
# Cell 14: Step 6 — 약어 정규화 후 unmatched 재매칭
ABBREV = [
    (r'\bcorporation\b', 'corp'),
    (r'\bincorporated\b', 'inc'),
    (r'\blimited\b', 'ltd'),
    (r'\bcompany\b', 'co'),
]

def abbrev_norm(s):
    if pd.isna(s): return ''
    s = str(s).strip().lower()
    s = re.sub(r'[,\.\-\(\)\/\\&\'"]+', ' ', s)
    for pattern, repl in ABBREV:
        s = re.sub(pattern, repl, s)
    s = re.sub(r'\s{2,}', ' ', s).strip()
    return s

def match_abbrev_all(assignees, party_str):
    if not isinstance(assignees, list) or not assignees or pd.isna(party_str):
        return '', ''
    party_map = {abbrev_norm(p): p.strip() for p in party_str.split(' | ')}
    party_map.pop('', None)
    for org in assignees:
        if pd.isna(org): continue
        key = abbrev_norm(org)
        if key and key in party_map:
            return org.strip(), party_map[key]
    return '', ''

still_unmatched = case_pat['match_src'] == 'unmatched'

# g_assignee + chain 후보 합쳐서 시도
def combined_assignees(r):
    a = r['assignees'] if isinstance(r['assignees'], list) else []
    b = r['chain_assignees'] if isinstance(r['chain_assignees'], list) else []
    seen = set()
    result = []
    for x in a + b:
        if x and x not in seen:
            seen.add(x)
            result.append(x)
    return result if result else None

case_pat.loc[still_unmatched, 'combined'] = case_pat[still_unmatched].apply(combined_assignees, axis=1)

abbrev_results = case_pat[still_unmatched].apply(
    lambda r: pd.Series({
        'abbrev_pla_match':  match_abbrev_all(r['combined'], r['plaintiffs'])[1],
        'abbrev_def_match':  match_abbrev_all(r['combined'], r['defendants'])[1],
        'abbrev_owner_pla':  match_abbrev_all(r['combined'], r['plaintiffs'])[0],
        'abbrev_owner_def':  match_abbrev_all(r['combined'], r['defendants'])[0],
    }), axis=1
)
for col in abbrev_results.columns:
    case_pat[col] = abbrev_results[col].reindex(case_pat.index).fillna('')

case_pat.loc[still_unmatched & (case_pat['abbrev_pla_match'] != ''),
             'match_src'] = 'abbrev_plaintiff'
case_pat.loc[still_unmatched & (case_pat['abbrev_pla_match'] == '') &
             (case_pat['abbrev_def_match'] != ''),
             'match_src'] = 'abbrev_defendant'

total = len(case_pat)
n_pla = (case_pat['match_src'] == 'abbrev_plaintiff').sum()
n_def = (case_pat['match_src'] == 'abbrev_defendant').sum()
print(f'약어 정규화 추가 매칭: 원고 {n_pla:,}행 + 피고 {n_def:,}행 = {n_pla+n_def:,}행')
matched_all = case_pat['match_src'].str.contains('plaintiff|defendant', na=False).sum()
print(f'전체 누계 매칭: {matched_all:,}행 ({matched_all/total:.1%})')

약어 정규화 추가 매칭: 원고 1,865행 + 피고 108행 = 1,973행
전체 누계 매칭: 71,390행 (70.0%)


In [83]:
# Cell 15: patent_ownership_matched_3.csv 저장
case_pat['final_owner']      = ''
case_pat['matched_assignee'] = ''

src_map = {
    'raw_plaintiff':              ('raw_pla_match',              'raw_matched_assignee'),
    'raw_defendant':              ('raw_def_match',               'raw_matched_assignee_d'),
    'simple_plaintiff':           ('simple_pla_match',            'simple_matched_assignee'),
    'simple_defendant':           ('simple_def_match',            'simple_matched_assignee_d'),
    'chain_raw_plaintiff':        ('chain_raw_pla_match',         'chain_raw_owner_pla'),
    'chain_raw_defendant':        ('chain_raw_def_match',         'chain_raw_owner_def'),
    'chain_simple_plaintiff':     ('chain_simple_pla_match',      'chain_simple_owner_pla'),
    'chain_simple_defendant':     ('chain_simple_def_match',      'chain_simple_owner_def'),
    'na_chain_raw_plaintiff':     ('na_chain_raw_pla_match',      'na_chain_raw_owner_pla'),
    'na_chain_raw_defendant':     ('na_chain_raw_def_match',      'na_chain_raw_owner_def'),
    'na_chain_simple_plaintiff':  ('na_chain_simple_pla_match',   'na_chain_simple_owner_pla'),
    'abbrev_defendant':           ('abbrev_def_match',            'abbrev_owner_def'),
}
for src, (party_col, owner_col) in src_map.items():
    mask = case_pat['match_src'] == src
    if mask.any() and party_col in case_pat.columns:
        case_pat.loc[mask, 'final_owner']      = case_pat.loc[mask, party_col]
        case_pat.loc[mask, 'matched_assignee'] = case_pat.loc[mask, owner_col]

case_pat['owner_side'] = ''
case_pat.loc[case_pat['match_src'].str.contains('plaintiff', na=False), 'owner_side'] = 'P'
case_pat.loc[case_pat['match_src'].str.contains('defendant', na=False), 'owner_side'] = 'D'

out_cols = ['case_row_id', 'date_filed', 'date_last_filed', 'patent_norm',
            'matched_assignee', 'owner_side', 'final_owner', 'match_src',
            'plaintiffs', 'defendants']
out_path = OUT_DIR / 'patent_ownership_matched_3.csv'
case_pat[out_cols].to_csv(out_path, index=False)

total = len(case_pat)
matched = case_pat['match_src'].str.contains('plaintiff|defendant', na=False).sum()
print(f'저장 완료: {out_path}')
print(f'전체 매칭: {matched:,}행 ({matched/total:.1%})')
print(f'최종 미매칭: {(case_pat["match_src"] == "unmatched").sum():,}행')
print(f'g_assignee 없음: {(case_pat["match_src"] == "no_assignee").sum():,}행')

저장 완료: ../data/samples/patent_ownership_matched_3.csv
전체 매칭: 71,390행 (70.0%)
최종 미매칭: 20,130행
g_assignee 없음: 10,398행


In [84]:
still_un = case_pat[case_pat['match_src'] == 'unmatched'].copy()
still_un['g_assignee_owners'] = still_un['assignees'].apply(
    lambda x: ' | '.join(x) if isinstance(x, list) else ''
)
still_un['chain_owners'] = still_un['chain_assignees'].apply(
    lambda x: ' | '.join(x) if isinstance(x, list) else ''
)
still_un[['case_row_id','patent_norm','g_assignee_owners','chain_owners','plaintiffs','defendants']].head(30)

,case_row_id,patent_norm,g_assignee_owners,chain_owners,plaintiffs,defendants
12,742,6411942,Fujitsu Limited,FUJITSU LIMITED,"Firepond, Inc. | a Delaware corporation","Banter, Inc."
19,750,5196040,Dundee Manufacturing Co. Inc. | Grease Guard LLC,"GREASE GUARD, INC.","Ecolab, Inc. | Facilitec Corp.","Omni Containment Systems, LLC"
22,753,6312197,"Anchor Wall Systems, Inc.",,Anchor Wall Systems | a Minnesota corporation,"Concrete Products of New London, Inc."
25,757,6032723,Calsonic Corporation,CALSONIC CORPORATION,Select Comfort Corporation,"Halcyon Waterspring, Inc."
35,774,5277849,Dri-Steam Humidifier,RESEARCH PRODUCTS CORPORATION,Dri-Steem Humidifier Company,Carel USA
44,785,6395702,"Sunburst Chemicals, Inc.",,"Suing as Sunburst Chemicals, Inc., a Minnesota...",Noramtech Corporation | a Missouri corporation
49,790,6361499,"CIVCO Medical Instruments Co., Inc.",CIVCO MEDICAL INSTRUMENTS INC.,"CIVCO Medical Insatruments Co., Inc.","Protek Medical Products, Inc."
51,794,6293475,"Summers Manufacturing Company, Inc.","SUMMERS MANUFACTURING CO., INC.","Summers Manufacturing, Inc.",Spray-Air Technologies Ltd.
59,803,5006344,E.R. SQUIBB & SONS. INC.,"E.R. SQUIBB & SONS, INC. | E.R. SQUIBB & SONS,...",Bristol-Meyers Squibb Company | E.R. Squibb & ...,Andrx Pharmaceuticals L.L.C | Andrx Pharmaceut...
62,807,4654217,Tadaaki Sakai,"TADAAKI SAKIA, 39-8 YAMADANISHI 3-CHOME, SUITA...","TruFresh, LLC | Winterlab, Ltd.","Atlantic Trading Co. East, Inc. | Atlantic Tra..."


In [85]:
# Cell 16: suffix 제거 후 재매칭
SUFFIXES = r'\b(inc|corp|co|ltd|llc|lp|llp|plc|incorporated|corporation|limited|company|' \
           r'limited liability company|gmbh|bv|ag|sa|nv|oy|ab|srl|sas)\b'

def strip_norm(s):
    if pd.isna(s): return ''
    s = str(s).strip().lower()
    s = re.sub(r'[,\.\-\(\)\/\\&\'"]+', ' ', s)
    s = re.sub(SUFFIXES, ' ', s)
    s = re.sub(r'\s{2,}', ' ', s).strip()
    return s

def match_strip_all(assignees, party_str):
    if not isinstance(assignees, list) or not assignees or pd.isna(party_str):
        return '', ''
    party_map = {strip_norm(p): p.strip() for p in party_str.split(' | ')}
    party_map.pop('', None)
    for org in assignees:
        if pd.isna(org): continue
        key = strip_norm(org)
        if key and key in party_map:
            return org.strip(), party_map[key]
    return '', ''

still_un2 = case_pat['match_src'] == 'unmatched'
case_pat.loc[still_un2, 'combined'] = case_pat[still_un2].apply(combined_assignees, axis=1)

strip_results = case_pat[still_un2].apply(
    lambda r: pd.Series({
        'strip_pla_match': match_strip_all(r['combined'], r['plaintiffs'])[1],
        'strip_def_match': match_strip_all(r['combined'], r['defendants'])[1],
        'strip_owner_pla': match_strip_all(r['combined'], r['plaintiffs'])[0],
        'strip_owner_def': match_strip_all(r['combined'], r['defendants'])[0],
    }), axis=1
)
for col in strip_results.columns:
    case_pat[col] = strip_results[col].reindex(case_pat.index).fillna('')

case_pat.loc[still_un2 & (case_pat['strip_pla_match'] != ''),
             'match_src'] = 'strip_plaintiff'
case_pat.loc[still_un2 & (case_pat['strip_pla_match'] == '') &
             (case_pat['strip_def_match'] != ''),
             'match_src'] = 'strip_defendant'

total = len(case_pat)
n_pla = (case_pat['match_src'] == 'strip_plaintiff').sum()
n_def = (case_pat['match_src'] == 'strip_defendant').sum()
print(f'suffix 제거 추가 매칭: 원고 {n_pla:,}행 + 피고 {n_def:,}행 = {n_pla+n_def:,}행')
matched_all = case_pat['match_src'].str.contains('plaintiff|defendant', na=False).sum()
print(f'전체 누계 매칭: {matched_all:,}행 ({matched_all/total:.1%})') 

suffix 제거 추가 매칭: 원고 4,230행 + 피고 128행 = 4,358행
전체 누계 매칭: 75,748행 (74.3%)


In [86]:
# 매칭된 예시 확인
ex_pla = case_pat[case_pat['match_src'] == 'strip_plaintiff'][
    ['patent_norm', 'strip_owner_pla', 'strip_pla_match', 'plaintiffs', 'defendants']
].head(15)
print('=== 원고 매칭 예시 ===')
print(ex_pla.to_string(index=False))

ex_def = case_pat[case_pat['match_src'] == 'strip_defendant'][
    ['patent_norm', 'strip_owner_def', 'strip_def_match', 'plaintiffs', 'defendants']
].head(15)
print('\n=== 피고 매칭 예시 ===')
print(ex_def.to_string(index=False))

=== 원고 매칭 예시 ===
patent_norm                     strip_owner_pla                        strip_pla_match                                                                                                                                                                                 plaintiffs                                                                                                                                                                                                                                                                                             defendants
    6312197           Anchor Wall Systems, Inc.                    Anchor Wall Systems                                                                                                                                              Anchor Wall Systems | a Minnesota corporation                                                                                                                                             

In [87]:

still_un3 = case_pat[case_pat['match_src'] == 'unmatched'].copy()
still_un3['g_assignee_owners'] = still_un3['assignees'].apply(
lambda x: ' | '.join(x) if isinstance(x, list) else ''
)
still_un3['chain_owners'] = still_un3['chain_assignees'].apply(
lambda x: ' | '.join(x) if isinstance(x, list) else ''
)
# assignee 있는 것만 (no_assignee 제외)
has_owner = still_un3[(still_un3['g_assignee_owners'] != '') | (still_un3['chain_owners'] != '')]
print(f'assignee 있는 미매칭: {len(has_owner):,}행')
has_owner[['patent_norm','g_assignee_owners','chain_owners','plaintiffs','defendants']].head(30)

assignee 있는 미매칭: 15,772행


,patent_norm,g_assignee_owners,chain_owners,plaintiffs,defendants
12,6411942,Fujitsu Limited,FUJITSU LIMITED,"Firepond, Inc. | a Delaware corporation","Banter, Inc."
19,5196040,Dundee Manufacturing Co. Inc. | Grease Guard LLC,"GREASE GUARD, INC.","Ecolab, Inc. | Facilitec Corp.","Omni Containment Systems, LLC"
25,6032723,Calsonic Corporation,CALSONIC CORPORATION,Select Comfort Corporation,"Halcyon Waterspring, Inc."
35,5277849,Dri-Steam Humidifier,RESEARCH PRODUCTS CORPORATION,Dri-Steem Humidifier Company,Carel USA
49,6361499,"CIVCO Medical Instruments Co., Inc.",CIVCO MEDICAL INSTRUMENTS INC.,"CIVCO Medical Insatruments Co., Inc.","Protek Medical Products, Inc."
62,4654217,Tadaaki Sakai,"TADAAKI SAKIA, 39-8 YAMADANISHI 3-CHOME, SUITA...","TruFresh, LLC | Winterlab, Ltd.","Atlantic Trading Co. East, Inc. | Atlantic Tra..."
64,5725564,ESD Limited Liability Company,ESD LIMITED LIABILITY COMPANY,"ESD, LLC | Encore Medical, LP | Vitalstim, LLC",Teresa P. Biber
70,5725564,ESD Limited Liability Company,ESD LIMITED LIABILITY COMPANY,"ESD, LLC | Encore Medical, LP | Vitalstim, LLC",Communication and Swallowing Rehabilitation Se...
96,5400020,"Global Research Systems, Inc.","ARRIVALSTAR, INC.","Arrival Star, Inc. | a foreign corporation",Frontier Airlines | HighJump Software | Nistev...
108,5496605,Minnesota Mining and Manufacturing Company,,3M Company | 3M Innovative Properties Company,"Medline Industries, Inc."


In [88]:
# Cell 17: 최종 체인 소유자로 미매칭 재시도
still_un3 = case_pat[case_pat['match_src'] == 'unmatched'].copy()
has_owner_idx = still_un3[
    still_un3['assignees'].apply(lambda x: isinstance(x, list) and len(x) > 0) |
    still_un3['chain_assignees'].apply(lambda x: isinstance(x, list) and len(x) > 0)
].index

un_patents = case_pat.loc[has_owner_idx, 'patent_norm'].unique()

# chain_df에서 날짜 무관 최신 소유자
final_chain = (
    chain_df[chain_df['patent_norm'].isin(un_patents)]
    .sort_values('end_dt', ascending=False)
    .groupby('patent_norm')['ee_name']
    .first()
    .reset_index()
    .rename(columns={'ee_name': 'final_chain_name'})
)
print(f'최종 체인 소유자 있는 미매칭 특허: {final_chain["patent_norm"].nunique():,}개')

# case_pat에 합치기
tmp = case_pat.loc[has_owner_idx].merge(final_chain, on='patent_norm', how='left')
tmp.index = has_owner_idx

def match_final(r):
    candidates = []
    if isinstance(r['assignees'], list):
        candidates += r['assignees']
    fc = r.get('final_chain_name', '')
    if pd.notna(fc) and fc and fc not in candidates:
        candidates.append(fc)
    return candidates

tmp['final_candidates'] = tmp.apply(match_final, axis=1)

final_results = tmp.apply(
    lambda r: pd.Series({
        'final_pla_match': match_strip_all(r['final_candidates'], r['plaintiffs'])[1],
        'final_def_match': match_strip_all(r['final_candidates'], r['defendants'])[1],
        'final_owner_pla': match_strip_all(r['final_candidates'], r['plaintiffs'])[0],
        'final_owner_def': match_strip_all(r['final_candidates'], r['defendants'])[0],
    }), axis=1
)
for col in final_results.columns:
    case_pat[col] = final_results[col].reindex(case_pat.index).fillna('')

still_un3_mask = case_pat['match_src'] == 'unmatched'
case_pat.loc[still_un3_mask & (case_pat['final_pla_match'] != ''),
             'match_src'] = 'final_plaintiff'
case_pat.loc[still_un3_mask & (case_pat['final_pla_match'] == '') &
             (case_pat['final_def_match'] != ''),
             'match_src'] = 'final_defendant'
n_pla = (case_pat['match_src'] == 'final_plaintiff').sum()
n_def = (case_pat['match_src'] == 'final_defendant').sum()
print(f'최종 소유자 추가 매칭: 원고 {n_pla:,}행 + 피고 {n_def:,}행 = {n_pla+n_def:,}행')
matched_all = case_pat['match_src'].str.contains('plaintiff|defendant', na=False).sum()
print(f'전체 누계 매칭: {matched_all:,}행 ({matched_all/total:.1%})')

최종 체인 소유자 있는 미매칭 특허: 5,557개
최종 소유자 추가 매칭: 원고 800행 + 피고 50행 = 850행
전체 누계 매칭: 76,598행 (75.2%)


In [89]:
ex_pla = case_pat[case_pat['match_src'] == 'final_plaintiff'][
    ['patent_norm', 'final_owner_pla', 'final_pla_match', 'plaintiffs', 'defendants']
].head(15)
print('=== 원고 매칭 예시 ===')
print(ex_pla.to_string(index=False))

ex_def = case_pat[case_pat['match_src'] == 'final_defendant'][
    ['patent_norm', 'final_owner_def', 'final_def_match', 'plaintiffs', 'defendants']
].head(15)
print('\n=== 피고 매칭 예시 ===')
print(ex_def.to_string(index=False))

=== 원고 매칭 예시 ===
patent_norm                   final_owner_pla                   final_pla_match                                                                                                                plaintiffs                                                                                                                                                                                                                                                                                                                                                           defendants
    5612524              VCODE HOLDINGS, INC.                VCode Holdings Inc                                                     LLC | TERMINATED: 10/25/2006 | VCode Holdings Inc | VData | VData LLC Adidas America, Inc. | Adidas Salomon | Adidas-Salomon AG | Advanced Micro Devices, Inc. | Boston Scientific Corp. | Hitachi Global Storage Technologies (Thailand), Ltd. | Hitachi Global Storage Technologies, Inc. | Stamps

In [90]:
still_un4 = case_pat[case_pat['match_src'] == 'unmatched'].copy()
still_un4['g_assignee_owners'] = still_un4['assignees'].apply(
    lambda x: ' | '.join(x) if isinstance(x, list) else ''
)
still_un4['chain_owners'] = still_un4['chain_assignees'].apply(
    lambda x: ' | '.join(x) if isinstance(x, list) else ''
)
has_owner2 = still_un4[(still_un4['g_assignee_owners'] != '') | (still_un4['chain_owners'] != '')]
print(f'assignee 있는 미매칭: {len(has_owner2):,}행')
has_owner2[['patent_norm','g_assignee_owners','chain_owners','plaintiffs','defendants']].head(30)

assignee 있는 미매칭: 14,922행


,patent_norm,g_assignee_owners,chain_owners,plaintiffs,defendants
12,6411942,Fujitsu Limited,FUJITSU LIMITED,"Firepond, Inc. | a Delaware corporation","Banter, Inc."
19,5196040,Dundee Manufacturing Co. Inc. | Grease Guard LLC,"GREASE GUARD, INC.","Ecolab, Inc. | Facilitec Corp.","Omni Containment Systems, LLC"
25,6032723,Calsonic Corporation,CALSONIC CORPORATION,Select Comfort Corporation,"Halcyon Waterspring, Inc."
35,5277849,Dri-Steam Humidifier,RESEARCH PRODUCTS CORPORATION,Dri-Steem Humidifier Company,Carel USA
49,6361499,"CIVCO Medical Instruments Co., Inc.",CIVCO MEDICAL INSTRUMENTS INC.,"CIVCO Medical Insatruments Co., Inc.","Protek Medical Products, Inc."
62,4654217,Tadaaki Sakai,"TADAAKI SAKIA, 39-8 YAMADANISHI 3-CHOME, SUITA...","TruFresh, LLC | Winterlab, Ltd.","Atlantic Trading Co. East, Inc. | Atlantic Tra..."
64,5725564,ESD Limited Liability Company,ESD LIMITED LIABILITY COMPANY,"ESD, LLC | Encore Medical, LP | Vitalstim, LLC",Teresa P. Biber
70,5725564,ESD Limited Liability Company,ESD LIMITED LIABILITY COMPANY,"ESD, LLC | Encore Medical, LP | Vitalstim, LLC",Communication and Swallowing Rehabilitation Se...
96,5400020,"Global Research Systems, Inc.","ARRIVALSTAR, INC.","Arrival Star, Inc. | a foreign corporation",Frontier Airlines | HighJump Software | Nistev...
108,5496605,Minnesota Mining and Manufacturing Company,,3M Company | 3M Innovative Properties Company,"Medline Industries, Inc."


In [92]:
# Cell 18: patent_ownership_matched_4.csv 저장
case_pat['final_owner']      = ''
case_pat['matched_assignee'] = ''

src_map = {
    'raw_plaintiff':             ('raw_pla_match',            'raw_matched_assignee'),
    'raw_defendant':             ('raw_def_match',             'raw_matched_assignee_d'),
    'simple_plaintiff':          ('simple_pla_match',          'simple_matched_assignee'),
    'simple_defendant':          ('simple_def_match',          'simple_matched_assignee_d'),
    'chain_raw_plaintiff':       ('chain_raw_pla_match',       'chain_raw_owner_pla'),
    'chain_raw_defendant':       ('chain_raw_def_match',       'chain_raw_owner_def'),
    'chain_simple_plaintiff':    ('chain_simple_pla_match',    'chain_simple_owner_pla'),
    'chain_simple_defendant':    ('chain_simple_def_match',    'chain_simple_owner_def'),
    'na_chain_raw_plaintiff':    ('na_chain_raw_pla_match',    'na_chain_raw_owner_pla'),
    'na_chain_raw_defendant':    ('na_chain_raw_def_match',    'na_chain_raw_owner_def'),
    'na_chain_simple_plaintiff': ('na_chain_simple_pla_match', 'na_chain_simple_owner_pla'),
    'na_chain_simple_defendant': ('na_chain_simple_def_match', 'na_chain_simple_owner_def'),
    'abbrev_plaintiff':          ('abbrev_pla_match',          'abbrev_owner_pla'),
    'abbrev_defendant':          ('abbrev_def_match',          'abbrev_owner_def'),
    'strip_plaintiff':           ('strip_pla_match',           'strip_owner_pla'),
    'strip_defendant':           ('strip_def_match',           'strip_owner_def'),
    'final_plaintiff':           ('final_pla_match',           'final_owner_pla'),
    'final_defendant':           ('final_def_match',           'final_owner_def'),
}
for src, (party_col, owner_col) in src_map.items():
    mask = case_pat['match_src'] == src
    if mask.any() and party_col in case_pat.columns:
        case_pat.loc[mask, 'final_owner']      = case_pat.loc[mask, party_col]
        case_pat.loc[mask, 'matched_assignee'] = case_pat.loc[mask, owner_col]

case_pat['owner_side'] = ''
case_pat.loc[case_pat['match_src'].str.contains('plaintiff', na=False), 'owner_side'] = 'P'
case_pat.loc[case_pat['match_src'].str.contains('defendant', na=False), 'owner_side'] = 'D'

out_cols = ['case_row_id', 'date_filed', 'date_last_filed', 'patent_norm',
            'matched_assignee', 'owner_side', 'final_owner', 'match_src',
            'plaintiffs', 'defendants']
out_path = OUT_DIR / 'patent_ownership_matched_4.csv'
case_pat[out_cols].to_csv(out_path, index=False)

total = len(case_pat)
matched = case_pat['match_src'].str.contains('plaintiff|defendant', na=False).sum()
print(f'저장 완료: {out_path}')
print(f'전체 매칭: {matched:,}행 ({matched/total:.1%})')
print(f'최종 미매칭: {(case_pat["match_src"] == "unmatched").sum():,}행')
print(f'g_assignee 없음: {(case_pat["match_src"] == "no_assignee").sum():,}행')

저장 완료: ../data/samples/patent_ownership_matched_4.csv
전체 매칭: 76,598행 (75.2%)
최종 미매칭: 14,922행
g_assignee 없음: 10,398행


In [94]:
df = pd.read_csv('/home/jmbyeon/developments/tf_litigation/MANTF-master/data/samples/patent_ownership_matched_4.csv')

matched = df[df['owner_side'].isin(['P', 'D'])].copy()

case_sides = matched.groupby('case_row_id')['owner_side'].apply(set)
both_sides = case_sides[case_sides.apply(lambda s: 'P' in s and 'D' in s)]

print(f'P/D 둘 다 매칭된 case 수: {len(both_sides):,}건')
print(f'해당 case의 특허 행 수: {matched[matched["case_row_id"].isin(both_sides.index)].shape[0]:,}행')

both_ex = matched[matched['case_row_id'].isin(both_sides.index[:5])][
    ['case_row_id', 'patent_norm', 'owner_side', 'final_owner', 'plaintiffs', 'defendants']
].sort_values(['case_row_id', 'owner_side'])
print(both_ex.to_string(index=False))

P/D 둘 다 매칭된 case 수: 269건
해당 case의 특허 행 수: 1,460행
 case_row_id patent_norm owner_side                                   final_owner                                      plaintiffs                                                                                                                                                                                                                                            defendants
         731     5800460          D          Koninklijke Philips Electronics N.V.   Cardiac Science, Inc | a Delaware Corporation Koninklijke Philips Electronics N.V. | Koninklijke Philips Electronics NV | Philips Electronics North America Corporation | Philips Medical Systems North America Company | Royal Philips Electronics | a Netherlands corporation | doing business as
         731     6016059          D          Koninklijke Philips Electronics N.V.   Cardiac Science, Inc | a Delaware Corporation Koninklijke Philips Electronics N.V. | Koninklijke Philips Electronic

In [95]:


df = pd.read_csv('/home/jmbyeon/developments/tf_litigation/MANTF-master/data/samples/patent_ownership_matched_4.csv')

matched = df[df['owner_side'].isin(['P', 'D'])]
case_sides = matched.groupby('case_row_id')['owner_side'].apply(set)
both_ids = case_sides[case_sides.apply(lambda s: 'P' in s and 'D' in s)].index

both_df = matched[matched['case_row_id'].isin(both_ids)]

p_patents = both_df[both_df['owner_side'] == 'P'].groupby('case_row_id')['patent_norm'].apply(lambda x: ' | '.join(x)).rename('P_patents')
d_patents = both_df[both_df['owner_side'] == 'D'].groupby('case_row_id')['patent_norm'].apply(lambda x: ' | '.join(x)).rename('D_patents')
parties   = both_df.groupby('case_row_id')[['plaintiffs', 'defendants']].first()

result = parties.join(p_patents).join(d_patents).reset_index()
result = result[['case_row_id', 'P_patents', 'D_patents', 'plaintiffs', 'defendants']]

print(f'P/D 둘 다 매칭된 case 수: {len(result):,}건')
out = '/home/jmbyeon/developments/tf_litigation/MANTF-master/data/samples/both_side_patents.csv'
result.to_csv(out, index=False)
print(f'저장 완료: {out}')
print(result.head(5).to_string(index=False))

P/D 둘 다 매칭된 case 수: 269건
저장 완료: /home/jmbyeon/developments/tf_litigation/MANTF-master/data/samples/both_side_patents.csv
 case_row_id                                                                                         P_patents                                                                                                   D_patents                                      plaintiffs                                                                                                                                                                                                                                            defendants
         731 5897576 | 6088616 | 6366809 | 5700281 | 5645571 | 5797969 | 6029085 | 5579919 | 5984102 | 5402884 5800460 | 6016059 | 5591213 | 5773961 | 6230054 | 5879374 | 5868792 | 5904707 | 6047212 | 5899926 | 5607454   Cardiac Science, Inc | a Delaware Corporation Koninklijke Philips Electronics N.V. | Koninklijke Philips Electronics NV | Philips Electronics Nor

In [96]:
df = pd.read_csv('/home/jmbyeon/developments/tf_litigation/MANTF-master/data/samples/patent_ownership_matched_4.csv')

n_def_cases = df[df['owner_side'] == 'D']['case_row_id'].nunique()
n_def_rows  = (df['owner_side'] == 'D').sum()
print(f'피고 특허 매칭된 case 수: {n_def_cases:,}건')
print(f'피고 특허 매칭된 행 수: {n_def_rows:,}행')

피고 특허 매칭된 case 수: 759건
피고 특허 매칭된 행 수: 1,894행


In [98]:
# D만 있는 케이스의 소송 유형 샘플
d_only = df.groupby('case_row_id')['owner_side'].apply(set)
d_only_ids = d_only[d_only.apply(lambda s: s == {'D'})].index
print(f'D만 있는 케이스: {len(d_only_ids):,}건')
df[df['case_row_id'].isin(d_only_ids)][['case_row_id','plaintiffs','defendants','patent_norm']].head(10)

D만 있는 케이스: 344건


,case_row_id,plaintiffs,defendants,patent_norm
14,745,IMI Cornelius Inc.,Lancer Corporation | Lancer Partnership Ltd.,4916910
107,859,Tassilo Bonzel,Boston Scientific Corporation | Boston Scienti...,4762129
181,949,"Cut Kleen Chem, Inc. | Lee C Pedersen | Releas...","Akona, LLC | Kleen Products, LLC | Stephen H H...",6083891
232,1004,Globetel Communications Corp | a Delaware Corp...,"Alexsam, Inc | John Doe 1 | John Doe 2 | Rober...",6000608
362,1165,Imation Corporation | a Delaware Corporation,Koninklijke Philips Electronics N.V. | Moser B...,6538982
371,1178,Muller Martini Corp,"Goss International Americas, Inc | Goss Intern...",5772195
423,1242,Edgar Newberg,"Michael Schweiss | Schweiss Distributing, Inc.",6199617
468,1297,Shawn McCutcheon,"Vaughn Neher Technology, LLC",7140262
498,1329,"Apotex Corp. | Apotex, Inc. | a Canadian corpo...",Bristol-Myers Squibb Company | Bristol-Myers S...,4847265
513,1353,Dyneon LLC | E.I. Du Pont de Numours and Company,3M Company | 3M Innovative Properties Company ...,7358296


In [99]:
df = pd.read_csv('/home/jmbyeon/developments/tf_litigation/MANTF-master/data/samples/patent_ownership_matched_4.csv')

matched = df[df['owner_side'].isin(['P', 'D'])]
p_only_ids   = case_sides[case_sides.apply(lambda s: s == {'P'})].index
d_only_ids   = case_sides[case_sides.apply(lambda s: s == {'D'})].index
both_ids     = case_sides[case_sides.apply(lambda s: 'P' in s and 'D' in s)].index

print('=== 케이스 수 ===')
print(f'원고만 있는 케이스:     {len(p_only_ids):,}건')
print(f'피고만 있는 케이스:     {len(d_only_ids):,}건')
print(f'원고+피고 둘 다 있는:  {len(both_ids):,}건')
print(f'합계:                   {len(p_only_ids)+len(d_only_ids)+len(both_ids):,}건')

print('\n=== 행 수 ===')
p_only_rows = matched[matched['case_row_id'].isin(p_only_ids)].shape[0]
d_only_rows = matched[matched['case_row_id'].isin(d_only_ids)].shape[0]
both_rows   = matched[matched['case_row_id'].isin(both_ids)].shape[0]
print(f'원고만 있는 행:         {p_only_rows:,}행')
print(f'피고만 있는 행:         {d_only_rows:,}행')
print(f'원고+피고 둘 다 있는:  {both_rows:,}행')
print(f'합계:                   {p_only_rows+d_only_rows+both_rows:,}행')
print(f'\n전체 매칭 행(참고):     {len(matched):,}행')

=== 케이스 수 ===
원고만 있는 케이스:     34,476건
피고만 있는 케이스:     490건
원고+피고 둘 다 있는:  269건
합계:                   35,235건

=== 행 수 ===
원고만 있는 행:         73,945행
피고만 있는 행:         1,193행
원고+피고 둘 다 있는:  1,460행
합계:                   76,598행

전체 매칭 행(참고):     76,598행


In [100]:
print(matched[matched['case_row_id'].isin(both_ids)]['owner_side'].value_counts())

owner_side
P    759
D    701
Name: count, dtype: int64


In [101]:
both_matched = matched[matched['case_row_id'].isin(both_ids)]
print(both_matched['owner_side'].value_counts())
print(f'\nP 보유 유니크 케이스: {both_matched[both_matched["owner_side"]=="P"]["case_row_id"].nunique():,}건')
print(f'D 보유 유니크 케이스: {both_matched[both_matched["owner_side"]=="D"]["case_row_id"].nunique():,}건')
print(f'(둘 다 동일하면 정상 = {len(both_ids):,}건)')

owner_side
P    759
D    701
Name: count, dtype: int64

P 보유 유니크 케이스: 269건
D 보유 유니크 케이스: 269건
(둘 다 동일하면 정상 = 269건)


In [102]:
p_patents = matched[matched['owner_side'] == 'P'].groupby('case_row_id')['patent_norm'].apply(lambda x: ' | '.join(x)).rename('P_patents')
d_patents = matched[matched['owner_side'] == 'D'].groupby('case_row_id')['patent_norm'].apply(lambda x: ' | '.join(x)).rename('D_patents')
parties   = df.groupby('case_row_id')[['plaintiffs', 'defendants']].first()

result = parties.join(p_patents).join(d_patents)
result = result[result.index.isin(both_ids)].reset_index()
result = result[['case_row_id', 'P_patents', 'D_patents', 'plaintiffs', 'defendants']]

out = '/home/jmbyeon/developments/tf_litigation/MANTF-master/data/samples/both_side_patents.csv'
result.to_csv(out, index=False)
print(f'저장 완료: {out}')
print(f'총 {len(result):,}건')
print(result.head(3).to_string(index=False))

저장 완료: /home/jmbyeon/developments/tf_litigation/MANTF-master/data/samples/both_side_patents.csv
총 269건
 case_row_id                                                                                         P_patents                                                                                                   D_patents                                    plaintiffs                                                                                                                                                                                                                                            defendants
         731 5897576 | 6088616 | 6366809 | 5700281 | 5645571 | 5797969 | 6029085 | 5579919 | 5984102 | 5402884 5800460 | 6016059 | 5591213 | 5773961 | 6230054 | 5879374 | 5868792 | 5904707 | 6047212 | 5899926 | 5607454 Cardiac Science, Inc | a Delaware Corporation Koninklijke Philips Electronics N.V. | Koninklijke Philips Electronics NV | Philips Electronics North America Corporation

In [4]:
df = pd.read_csv('/home/jmbyeon/developments/tf_litigation/MANTF-master/data/samples/patent_ownership_matched_4.csv')
df['patent_norm'].nunique()


37806